# Retail Analytics & AI-Powered Sales Forecasting System

**Goal:** analyze retail inventory/sales behavior, identify stockout risk, segment stores, and forecast future revenue.

This notebook is designed for **Google Colab** and uses the uploaded retail dimension/fact CSV files.


## 1. Project Objectives
- Analyze sales performance across stores, SKUs and categories
- Identify sales patterns and stockout/availability issues
- Study festive vs non-festive demand
- Segment stores using performance and behavior
- Build a time-series revenue forecasting model
- Produce management-ready insights and recommendations


In [ ]:
# CELL 1 — Install/import libraries
!pip -q install plotly scikit-learn statsmodels openpyxl

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Libraries loaded successfully.")


In [ ]:
# CELL 2 — Load the uploaded datasets
EVENTS = "/mnt/data/dim_events.csv"
SKUS = "/mnt/data/dim_skus.csv"
STORES = "/mnt/data/dim_stores.csv"
SUPPLIERS = "/mnt/data/dim_suppliers.csv"
INVENTORY = "/mnt/data/fact_inventory_daily.csv"

events = pd.read_csv(EVENTS, parse_dates=["date"])
skus = pd.read_csv(SKUS)
stores = pd.read_csv(STORES)
suppliers = pd.read_csv(SUPPLIERS)
inv = pd.read_csv(INVENTORY, parse_dates=["date"])

print("Events:", events.shape)
print("SKUs:", skus.shape)
print("Stores:", stores.shape)
print("Suppliers:", suppliers.shape)
print("Inventory:", inv.shape)


In [ ]:
# CELL 3 — Basic data quality checks
print("Date range:", inv["date"].min().date(), "to", inv["date"].max().date())
print("\nMissing values:")
display(inv.isna().sum().to_frame("missing").sort_values("missing", ascending=False))

print("\nDuplicate rows:", inv.duplicated().sum())
print("\nStockout risk distribution:")
display(inv["stockout_risk"].value_counts(dropna=False).to_frame("count"))


## 2. Data Preparation & Feature Engineering

We join the fact table with SKU, store, supplier and event information so that revenue and business KPIs can be analyzed at product/store/date level.


In [ ]:
# CELL 4 — Join dimension tables and create business features
df = inv.merge(
    skus[["sku_id","sku_name","category","unit_price_inr","shelf_life_days",
          "is_perishable","festive_relevant","popularity_tier","supplier_id"]],
    on=["sku_id","supplier_id"], how="left", suffixes=("","_sku")
)

df = df.merge(
    stores[["store_id","city","city_tier","store_size","sqft","opened_year","baseline_daily_orders"]],
    on="store_id", how="left"
)

df = df.merge(
    suppliers[["supplier_id","supplier_name","reliability_score",
               "base_lead_time_days","lead_time_variance_days"]],
    on="supplier_id", how="left"
)

df = df.merge(
    events[["date","event_name","event_type","demand_multiplier_festive","demand_multiplier_other"]],
    on="date", how="left"
)

df["revenue"] = df["units_sold"] * df["unit_price_inr"]
df["lost_units"] = (df["units_demanded"] - df["units_sold"]).clip(lower=0)
df["estimated_lost_revenue"] = df["lost_units"] * df["unit_price_inr"]
df["stockout_flag"] = (df["stockout_risk"].astype(str).str.lower() != "safe").astype(int)
df["reorder_flag"] = (df["reorder_placed"].astype(str).str.upper() == "Y").astype(int)
df["month"] = df["date"].dt.to_period("M").astype(str)
df["day_of_week"] = df["date"].dt.day_name()
df["week"] = df["date"].dt.isocalendar().week.astype(int)

print("Prepared dataset:", df.shape)
display(df.head())


## 3. KPI Summary

In [ ]:
# CELL 5 — Main KPIs
kpis = {
    "Total Revenue (₹)": df["revenue"].sum(),
    "Units Sold": df["units_sold"].sum(),
    "Units Demanded": df["units_demanded"].sum(),
    "Estimated Lost Revenue (₹)": df["estimated_lost_revenue"].sum(),
    "Stockout-risk Records": int(df["stockout_flag"].sum()),
    "Stores": df["store_id"].nunique(),
    "SKUs": df["sku_id"].nunique(),
    "Categories": df["category"].nunique()
}
display(pd.Series(kpis, name="Value").to_frame())


## 4. Sales & Revenue Analysis

In [ ]:
# CELL 6 — Daily revenue trend
daily = df.groupby("date", as_index=False).agg(
    revenue=("revenue","sum"),
    units_sold=("units_sold","sum"),
    units_demanded=("units_demanded","sum"),
    lost_units=("lost_units","sum")
)

fig = px.line(daily, x="date", y="revenue", title="Daily Revenue Trend")
fig.update_layout(xaxis_title="Date", yaxis_title="Revenue (₹)")
fig.show()


In [ ]:
# CELL 7 — Revenue by category
cat = df.groupby("category", as_index=False).agg(
    revenue=("revenue","sum"),
    units_sold=("units_sold","sum"),
    lost_revenue=("estimated_lost_revenue","sum")
).sort_values("revenue", ascending=False)

display(cat)
px.bar(cat, x="category", y="revenue", title="Revenue by Category").show()


In [ ]:
# CELL 8 — Revenue by store/city
store_perf = df.groupby(["store_id","city"], as_index=False).agg(
    revenue=("revenue","sum"),
    units_sold=("units_sold","sum"),
    lost_revenue=("estimated_lost_revenue","sum"),
    avg_days_cover=("days_of_cover","mean"),
    stockout_risk_rate=("stockout_flag","mean")
).sort_values("revenue", ascending=False)

display(store_perf)
px.bar(store_perf, x="store_id", y="revenue", color="city",
       title="Revenue by Store").show()


In [ ]:
# CELL 9 — Top 15 SKUs
sku_perf = df.groupby(["sku_id","sku_name","category"], as_index=False).agg(
    revenue=("revenue","sum"),
    units_sold=("units_sold","sum"),
    lost_revenue=("estimated_lost_revenue","sum")
).sort_values("revenue", ascending=False)

display(sku_perf.head(15))
px.bar(sku_perf.head(15), x="sku_name", y="revenue",
       color="category", title="Top 15 SKUs by Revenue").show()


## 5. Stockout Risk & Inventory Analysis

In [ ]:
# CELL 10 — Stockout risk analysis
risk = df.groupby("stockout_risk", as_index=False).agg(
    records=("stockout_risk","size"),
    lost_units=("lost_units","sum"),
    lost_revenue=("estimated_lost_revenue","sum")
).sort_values("lost_revenue", ascending=False)

display(risk)
px.bar(risk, x="stockout_risk", y="lost_revenue",
       title="Estimated Lost Revenue by Stockout Risk").show()


In [ ]:
# CELL 11 — Highest lost-revenue opportunities
lost = df.groupby(["store_id","city","sku_id","sku_name","category"], as_index=False).agg(
    lost_units=("lost_units","sum"),
    estimated_lost_revenue=("estimated_lost_revenue","sum"),
    avg_days_cover=("days_of_cover","mean"),
    avg_sales_velocity=("sales_velocity_7d","mean")
).sort_values("estimated_lost_revenue", ascending=False)

display(lost.head(20))


In [ ]:
# CELL 12 — Supplier performance
supplier_perf = df.groupby(["supplier_id","supplier_name"], as_index=False).agg(
    revenue=("revenue","sum"),
    reorder_count=("reorder_flag","sum"),
    avg_expected_lead=("lead_time_days_expected","mean"),
    avg_actual_lead=("lead_time_days_actual","mean"),
    reliability=("reliability_score","mean"),
    lost_revenue=("estimated_lost_revenue","sum")
).sort_values("lost_revenue", ascending=False)

display(supplier_perf)
px.scatter(supplier_perf, x="reliability", y="lost_revenue",
           size="reorder_count", hover_name="supplier_name",
           title="Supplier Reliability vs Estimated Lost Revenue").show()


## 6. Festive Demand Analysis

In [ ]:
# CELL 13 — Festival vs non-festival comparison
festive = df.groupby("is_festival_week", as_index=False).agg(
    revenue=("revenue","sum"),
    units_demanded=("units_demanded","sum"),
    units_sold=("units_sold","sum"),
    lost_revenue=("estimated_lost_revenue","sum")
)
festive["fill_rate"] = festive["units_sold"] / festive["units_demanded"].replace(0, np.nan)
display(festive)

px.bar(festive, x="is_festival_week", y="revenue",
       title="Revenue: Festival vs Non-Festival").show()


## 7. Store Segmentation with K-Means

Stores are grouped using revenue, units sold, lost revenue, stockout rate and inventory coverage.


In [ ]:
# CELL 14 — K-Means store segmentation
seg = df.groupby("store_id").agg(
    revenue=("revenue","sum"),
    units_sold=("units_sold","sum"),
    lost_revenue=("estimated_lost_revenue","sum"),
    stockout_rate=("stockout_flag","mean"),
    avg_days_cover=("days_of_cover","mean"),
    avg_sales_velocity=("sales_velocity_7d","mean")
).reset_index()

features = ["revenue","units_sold","lost_revenue","stockout_rate",
            "avg_days_cover","avg_sales_velocity"]

X = StandardScaler().fit_transform(seg[features])

n_clusters = min(4, len(seg))
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
seg["cluster"] = kmeans.fit_predict(X)

display(seg.sort_values(["cluster","revenue"], ascending=[True,False]))

px.scatter(seg, x="revenue", y="lost_revenue", color=seg["cluster"].astype(str),
           size="units_sold", hover_name="store_id",
           title="Store Segmentation").show()


## 8. Revenue Forecasting

Because the uploaded fact data contains daily observations, we aggregate revenue by date and train a time-series model. The last portion of the available period is held out for evaluation.


In [ ]:
# CELL 15 — Prepare daily revenue series
ts = daily.set_index("date")["revenue"].asfreq("D").fillna(0)
print(ts.head())
print("Observations:", len(ts))


In [ ]:
# CELL 16 — Holt-Winters forecast with evaluation
test_size = max(3, min(7, len(ts)//4))
train = ts.iloc[:-test_size]
test = ts.iloc[-test_size:]

# Weekly seasonality is appropriate for daily retail data when enough observations exist.
if len(train) >= 14:
    model = ExponentialSmoothing(
        train,
        trend="add",
        seasonal="add",
        seasonal_periods=7,
        initialization_method="estimated"
    ).fit(optimized=True)
else:
    model = ExponentialSmoothing(
        train, trend="add",
        initialization_method="estimated"
    ).fit(optimized=True)

pred = model.forecast(test_size)

mae = mean_absolute_error(test, pred)
rmse = mean_squared_error(test, pred) ** 0.5
print(f"MAE  : ₹{mae:,.2f}")
print(f"RMSE : ₹{rmse:,.2f}")

forecast_eval = pd.DataFrame({"Actual": test, "Forecast": pred})
display(forecast_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train, name="Train"))
fig.add_trace(go.Scatter(x=test.index, y=test, name="Actual"))
fig.add_trace(go.Scatter(x=pred.index, y=pred, name="Forecast"))
fig.update_layout(title="Revenue Forecast Evaluation", xaxis_title="Date", yaxis_title="Revenue (₹)")
fig.show()


In [ ]:
# CELL 17 — Forecast next 14 days
future_days = 14
future_forecast = model.forecast(future_days)

future_df = pd.DataFrame({
    "date": future_forecast.index,
    "forecast_revenue": future_forecast.values
})

display(future_df)

fig = px.line(future_df, x="date", y="forecast_revenue",
              markers=True, title="Next 14 Days Revenue Forecast")
fig.update_layout(yaxis_title="Forecast Revenue (₹)")
fig.show()


## 9. Machine Learning Benchmark

A Random Forest model is used as a second forecasting approach using calendar features and lagged revenue. This gives a simple machine-learning benchmark against the time-series model.


In [ ]:
# CELL 18 — Random Forest forecasting benchmark
ml = daily[["date","revenue"]].copy().sort_values("date")
ml["dow"] = ml["date"].dt.dayofweek
ml["day"] = ml["date"].dt.day
ml["month_num"] = ml["date"].dt.month
ml["is_weekend"] = (ml["dow"] >= 5).astype(int)

for lag in [1, 2, 3, 7]:
    ml[f"lag_{lag}"] = ml["revenue"].shift(lag)

ml = ml.dropna().reset_index(drop=True)

feature_cols = ["dow","day","month_num","is_weekend","lag_1","lag_2","lag_3","lag_7"]
split = int(len(ml) * 0.8)

train_ml = ml.iloc[:split]
test_ml = ml.iloc[split:]

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    max_depth=12,
    min_samples_leaf=2,
    n_jobs=-1
)
rf.fit(train_ml[feature_cols], train_ml["revenue"])
rf_pred = rf.predict(test_ml[feature_cols])

rf_mae = mean_absolute_error(test_ml["revenue"], rf_pred)
rf_rmse = mean_squared_error(test_ml["revenue"], rf_pred) ** 0.5
rf_r2 = r2_score(test_ml["revenue"], rf_pred)

print(f"Random Forest MAE  : ₹{rf_mae:,.2f}")
print(f"Random Forest RMSE : ₹{rf_rmse:,.2f}")
print(f"Random Forest R²   : {rf_r2:.3f}")

comparison = pd.DataFrame({
    "Model": ["Holt-Winters", "Random Forest"],
    "MAE": [mae, rf_mae],
    "RMSE": [rmse, rf_rmse]
})
display(comparison)


## 10. Automated Business Insights

In [ ]:
# CELL 19 — Generate management insights automatically
top_category = cat.iloc[0]
top_store = store_perf.iloc[0]
top_sku = sku_perf.iloc[0]
top_risk = lost.iloc[0]
highest_risk_level = risk.iloc[0]["stockout_risk"]

print("BUSINESS INSIGHTS")
print("="*70)
print(f"1. Top category by revenue: {top_category['category']} — ₹{top_category['revenue']:,.2f}")
print(f"2. Top store by revenue: {top_store['store_id']} ({top_store['city']}) — ₹{top_store['revenue']:,.2f}")
print(f"3. Top SKU by revenue: {top_sku['sku_name']} — ₹{top_sku['revenue']:,.2f}")
print(f"4. Highest estimated lost-revenue opportunity: {top_risk['sku_name']} at {top_risk['store_id']} — ₹{top_risk['estimated_lost_revenue']:,.2f}")
print(f"5. Most represented risk class by lost revenue: {highest_risk_level}")
print(f"6. Total estimated lost revenue: ₹{df['estimated_lost_revenue'].sum():,.2f}")
print(f"7. Forecast horizon: next {future_days} days")


## 11. Export Results

The following cell saves the key analytical outputs into an Excel workbook that can be used in a project submission or dashboard workflow.


In [ ]:
# CELL 20 — Export project outputs
output_file = "/content/Retail_Analytics_Project_Output.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    pd.DataFrame([kpis]).to_excel(writer, sheet_name="KPI_Summary", index=False)
    daily.to_excel(writer, sheet_name="Daily_Revenue", index=False)
    cat.to_excel(writer, sheet_name="Category_Performance", index=False)
    store_perf.to_excel(writer, sheet_name="Store_Performance", index=False)
    sku_perf.to_excel(writer, sheet_name="SKU_Performance", index=False)
    risk.to_excel(writer, sheet_name="Stockout_Risk", index=False)
    supplier_perf.to_excel(writer, sheet_name="Supplier_Performance", index=False)
    seg.to_excel(writer, sheet_name="Store_Segmentation", index=False)
    future_df.to_excel(writer, sheet_name="Revenue_Forecast", index=False)
    comparison.to_excel(writer, sheet_name="Model_Comparison", index=False)

print("Created:", output_file)


# Final Conclusion

This project provides an end-to-end retail analytics workflow:

- **Descriptive analytics:** revenue, units, category, store and SKU performance
- **Diagnostic analytics:** stockout risk, lost units, lost revenue and supplier performance
- **Predictive analytics:** daily revenue forecasting with Holt-Winters and a Random Forest benchmark
- **Prescriptive direction:** prioritize high lost-revenue SKU/store combinations, monitor low days-of-cover items, and plan inventory around festive demand
- **Management output:** KPI tables, interactive Plotly charts, store segmentation and an exportable Excel report

> **Important:** forecast accuracy depends on the amount and quality of historical data. With a short historical window, treat the forecast as a planning estimate rather than a guaranteed future value.
